# 04.8 — Weight initialization: He vs PyTorch defaults

A network's parameters start as random numbers — and *which* random distribution matters more than beginners expect. Pick it wrong and a deep network's activations vanish or explode before training even begins. This project **explicitly** sets He (Kaiming) initialization on its linear layers, rather than trusting PyTorch's default — because MATLAB's `'he'` initializer and PyTorch's `nn.Linear` default are *different distributions*, and matching them is a parity requirement. This notebook closes Module 04 on why.

This notebook covers:

* Why initialization matters — the vanishing/exploding-activation demo.
* He (Kaiming) init and the fan-in reasoning behind it.
* PyTorch's `nn.Linear` default and how it differs from MATLAB's `'he'`.
* Where the codebase calls `nn.init.kaiming_normal_` and why (Critical Note #31).

**Prerequisites:** [04.5 the bottleneck](04.5_the_bottleneck.ipynb), [02.4 tensors](../02_numpy_and_pytorch_basics/02.4_pytorch_tensors_intro.ipynb).

## Section 1 — What MATLAB does

MATLAB's fully-connected and conv layers take a `WeightsInitializer`:

```matlab
fullyConnectedLayer(HiddenSize, 'WeightsInitializer', 'he')
convolution1dLayer(FilterSize, NumFilters, 'WeightsInitializer', 'he')
```

`'he'` draws weights from `N(0, 2/fan_in)` — a normal distribution scaled by the number of inputs. It's the standard choice for ReLU-family networks (He et al. 2015). The MATLAB pipeline uses it on every learnable layer, so the Python port must reproduce that *exact* distribution — and PyTorch's default is not it.

## Section 2 — The Python concepts you need

### 2.1 — Why initialization matters at all

Stack many layers with a badly-scaled init and the activations shrink toward zero (or blow up) as they propagate — a network that can't learn because its signal died on the way in. Watch it across 20 layers, good init vs bad:

In [ ]:
import torch
from torch import nn

def activation_std_through_depth(gain):
    """Propagate a signal through 20 linear+ReLU layers; track activation std."""
    x = torch.randn(256, 128)
    stds = [x.std().item()]
    for _ in range(20):
        layer = nn.Linear(128, 128)
        nn.init.normal_(layer.weight, std=gain)     # deliberately mis-scaled
        nn.init.zeros_(layer.bias)
        x = torch.relu(layer(x))
        stds.append(x.std().item())
    return stds

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.6))
for gain, label in [(0.02, "std=0.02 (too small → vanish)"),
                    (0.30, "std=0.30 (too big → explode)"),
                    ((2/128)**0.5, "std=√(2/fan_in) = He (stable)")]:
    ax.semilogy(activation_std_through_depth(gain), marker="o", ms=3, label=label)
ax.set_xlabel("layer depth"); ax.set_ylabel("activation std (log)")
ax.legend(fontsize=8); ax.set_title("Init scale decides whether signal survives 20 layers")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

Too-small weights → activations decay toward zero (the gradient vanishes with them). Too-large → they blow up. The He scale, `√(2/fan_in)`, is tuned so a ReLU layer's output std stays ≈ its input std — signal (and gradient) propagate intact through depth. That "2" accounts for ReLU zeroing half its inputs.

### 2.2 — He initialization, explicitly

In [ ]:
torch.manual_seed(0)
layer = nn.Linear(128, 64)

# He (Kaiming) normal — the MATLAB 'he' distribution:
nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
nn.init.zeros_(layer.bias)

fan_in = 128
print(f"He target std: √(2/{fan_in}) = {(2/fan_in)**0.5:.4f}")
print(f"actual weight std after kaiming_normal_: {layer.weight.std().item():.4f}")

`nn.init.kaiming_normal_(w, nonlinearity="relu")` fills `w` in place ([02.4 §2.5](../02_numpy_and_pytorch_basics/02.4_pytorch_tensors_intro.ipynb)'s trailing-underscore convention) with `N(0, 2/fan_in)`. `fan_in` is the number of inputs (128 here). The measured std matches the target — this is exactly MATLAB's `'he'`.

### 2.3 — PyTorch's default is different

In [ ]:
torch.manual_seed(0)
default_layer = nn.Linear(128, 64)          # NO explicit init → PyTorch's default
he_layer = nn.Linear(128, 64)
nn.init.kaiming_normal_(he_layer.weight, nonlinearity="relu")

print(f"PyTorch default weight std: {default_layer.weight.std().item():.4f}")
print(f"He (MATLAB 'he')     std:   {he_layer.weight.std().item():.4f}")
print()
print("PyTorch's nn.Linear default is Kaiming UNIFORM with a=√5 — a DIFFERENT distribution")
print("(uniform not normal, and a different scale). Left implicit, it silently breaks MATLAB parity.")

PyTorch's `nn.Linear` default is `kaiming_uniform_(a=√5)` — a historical choice, uniform rather than normal, with a different effective scale. It's *fine* for training-from-scratch, but it is **not** MATLAB's `'he'`. So a port that leaves init implicit would start every layer from a different distribution than MATLAB — and while both might train, they wouldn't match, breaking the parity tests and any attempt to load MATLAB checkpoints into a freshly-built model. The fix is one explicit line per layer.

### 2.4 — Reproducibility ties in

Initialization is random, so it's governed by the seed ([00.8 §5](../00_orientation/00.8_build_a_dl_project_from_scratch.ipynb)). Same seed → same starting weights → reproducible training. Two freshly-built, He-initialized layers under the same seed are identical:

In [ ]:
torch.manual_seed(42); a = nn.Linear(64, 32); nn.init.kaiming_normal_(a.weight, nonlinearity="relu")
torch.manual_seed(42); b = nn.Linear(64, 32); nn.init.kaiming_normal_(b.weight, nonlinearity="relu")
print("same seed → identical init:", torch.equal(a.weight, b.weight))

## Section 3 — The `neural_data_decoding` implementation

Every explicit He init in the codebase looks the same — here it is in `LinearBottleneck` and `_EncoderBlock`, both citing Critical Note #31:

In [ ]:
from pathlib import Path

for path, marker in [
    ("../../src/neural_data_decoding/models/bottleneck.py", "kaiming_normal_"),
    ("../../src/neural_data_decoding/models/encoder.py", "kaiming_normal_"),
]:
    src = Path(path).read_text().splitlines()
    i = next(j for j, line in enumerate(src) if marker in line)
    print(f"# {path.split('/')[-1]}")
    for k in range(i - 1, min(i + 2, len(src))):
        print(f"{k + 1:4} | {src[k]}")
    print()

Things to spot:

* `nn.init.kaiming_normal_(weight, nonlinearity="relu")` immediately after every `nn.Linear` — the `relu` nonlinearity gives the `2/fan_in` scale (§2.1's factor of 2).
* `nn.init.zeros_(bias)` alongside — MATLAB starts biases at zero too.
* The pattern repeats in the classifier heads ([04.6](04.6_multi_head_classifier.ipynb)) and conv layers — everywhere a learnable weight is created. The comment cites Critical Note #31 so the *reason* (parity, not preference) is never lost.
* This is a recurring theme of the whole port: PyTorch's convenient defaults are frequently *not* MATLAB's, so the codebase overrides them explicitly and documents why — init here, `weight_decay` vs L2 in [02.7 §2.5](../02_numpy_and_pytorch_basics/02.7_optimizers_and_learning_rates.ipynb), the block order in [04.2](04.2_building_a_simple_encoder.ipynb).

## Section 4 — Hands-on exercises

### Exercise 4.1 — the fan-in scaling

He std is `√(2/fan_in)`. Build `Linear(256 → 10)` and `Linear(16 → 10)`, He-init both, and confirm the wider-input layer has *smaller* weights (larger fan_in → smaller std).

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
for fan_in in (256, 16):
    layer = nn.Linear(fan_in, 10)
    nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
    print(f"fan_in={fan_in:3}: target √(2/{fan_in})={ (2/fan_in)**0.5:.4f}, "
          f"actual std {layer.weight.std().item():.4f}")

### Exercise 4.2 — default vs He, quantified

Build 50 `Linear(100→100)` layers with PyTorch's default and 50 with He init; report the mean weight std of each group. How big is the gap?

In [ ]:
# Reveal:
import statistics
default_stds = [nn.Linear(100, 100).weight.std().item() for _ in range(50)]
he_stds = []
for _ in range(50):
    l = nn.Linear(100, 100); nn.init.kaiming_normal_(l.weight, nonlinearity="relu")
    he_stds.append(l.weight.std().item())
print(f"PyTorch default mean std: {statistics.mean(default_stds):.4f}")
print(f"He (MATLAB 'he')  mean std: {statistics.mean(he_stds):.4f}")
print(f"He is ~{statistics.mean(he_stds)/statistics.mean(default_stds):.1f}× wider — a real, systematic difference")

## Section 5 — Common errors

### MATLAB and Python diverge from step 0, before any training

Different initialization — the implicit-default trap (§2.3). Set He explicitly on every learnable layer; verify with the std check (§2.2).

### Deep model won't train; activations are ~0 or huge

Init scale (§2.1). For ReLU nets, `kaiming_normal_(nonlinearity="relu")`; for tanh/sigmoid, Xavier (`nn.init.xavier_normal_`) is the analog. Wrong nonlinearity arg → wrong scale.

### `kaiming_normal_` "does nothing"

It's in-place — `nn.init.kaiming_normal_(layer.weight)` modifies the weight; there's no return value to assign. Forgetting it (and writing `layer.weight = kaiming_normal_(...)`) is a mistake.

### Reproducibility broken across runs despite seeding

Init consumes RNG in construction order, so building modules in a different order (or building extra ones) shifts the RNG stream and changes all subsequent inits. Seed once, construct deterministically ([00.8 §5](../00_orientation/00.8_build_a_dl_project_from_scratch.ipynb)).

### Biases initialized randomly

MATLAB `'he'` zeros biases; PyTorch's default doesn't. The codebase adds `nn.init.zeros_(bias)` — omit it and biases start from a different distribution too.

## Section 6 — Further reading

- [He et al. 2015 — Delving Deep into Rectifiers](https://arxiv.org/abs/1502.01852) — the paper that introduced He init and the fan-in reasoning.
- [torch.nn.init docs](https://pytorch.org/docs/stable/nn.init.html) — every initializer, including the Xavier variants for non-ReLU nets.
- [Why PyTorch's default is kaiming_uniform(a=√5)](https://github.com/pytorch/pytorch/issues/15314) — the historical discussion, for the curious.

**Module 04 is complete.** You've walked the architecture end to end — the dispatcher, the encoders (simple, RNN, conv), the bottleneck, the multi-head classifier, the weighted loss, and the initialization that underlies them all. Module 05 (the training loop) is where this architecture starts learning. The [curriculum map](../README.md) tracks what's next.